# Modelado predictivo

En este notebook se prepara el dataset para entrenar modelos de Machine Learning.

El objetivo es predecir si una maquina puede fallar (`Machine failure`) usando variables operativas disponibles antes de conocer el fallo.

Se compararan varios modelos para elegir el que mejor detecte los fallos, prestando especial atencion al desbalance de clases.


In [3]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns


In [4]:
df = pd.read_csv("../data/raw/ai4i2020.csv")


In [5]:
df.shape

(10000, 14)

El dataset se carga correctamente y mantiene las dimensiones esperadas: 10000 registros y 14 columnas.


## Seleccion de variables

Para el modelo principal se usaran variables operativas que podrian estar disponibles antes de conocer si la maquina ha fallado.

Se excluyen identificadores como `UDI` y `Product ID`, ya que no describen el comportamiento real de la maquina.

Tambien se excluyen las columnas `TWF`, `HDF`, `PWF`, `OSF` y `RNF`, porque representan tipos concretos de fallo y podrian introducir fuga de informacion.


In [6]:
features = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

target = "Machine failure"


In [7]:
X = df[features]
y = df[target]


In [8]:
X.shape, y.shape


((10000, 6), (10000,))

In [9]:
X.head()


,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
0,M,298.1,308.6,1551,42.8,0
1,L,298.2,308.7,1408,46.3,3
2,L,298.1,308.5,1498,49.4,5
3,L,298.2,308.6,1433,39.5,7
4,L,298.2,308.7,1408,40.0,9


In [10]:
y.head()


0    0
1    0
2    0
3    0
4    0
Name: Machine failure, dtype: int64

Se crean `X` e `y` para separar las variables de entrada y la variable objetivo.

`X` contiene 10000 registros y 6 variables predictoras. `y` contiene la respuesta que el modelo intentara predecir.


## Separacion en entrenamiento y prueba

Se separan los datos en un conjunto de entrenamiento y un conjunto de prueba.

El conjunto de entrenamiento se usara para ajustar los modelos, mientras que el conjunto de prueba se reservara para evaluar su rendimiento con datos no vistos.

Como la variable objetivo esta desbalanceada, se usa una separacion estratificada para mantener una proporcion similar de fallos y no fallos en ambos conjuntos.


In [11]:
from sklearn.model_selection import train_test_split


In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=26,
    stratify=y
)


In [13]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape


((8000, 6), (2000, 6), (8000,), (2000,))

In [14]:
y_train.value_counts(normalize=True) * 100


Machine failure
0    96.6125
1     3.3875
Name: proportion, dtype: float64

In [15]:
y_test.value_counts(normalize=True) * 100


Machine failure
0    96.6
1     3.4
Name: proportion, dtype: float64

La separacion mantiene el 80% de los datos para entrenamiento y el 20% para prueba.

Se usa `random_state=26` para que la division sea reproducible. Gracias a `stratify=y`, la proporcion de maquinas con fallo y sin fallo se mantiene casi igual en ambos conjuntos.

Esto es importante porque el dataset esta muy desbalanceado y necesitamos que tanto entrenamiento como prueba representen bien el problema original.


## Preprocesamiento

Antes de entrenar los modelos, se prepara cada tipo de variable de forma adecuada.

La variable `Type` se transformara con `OneHotEncoder`, ya que es categorica. Las variables numericas se escalaran con `StandardScaler`, porque algunos modelos son sensibles a las diferencias de escala.

Para aplicar estas transformaciones de forma ordenada se usara `ColumnTransformer`.


In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [17]:
categorical_features = ["Type"]

numeric_features = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]


In [18]:
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", StandardScaler(), numeric_features)
    ]
)


## Modelo base: Regresion Logistica

Se empieza con una Regresion Logistica como primer modelo base de clasificacion.

Este modelo es sencillo, rapido y sirve como punto de comparacion frente a modelos mas complejos.

Como el dataset esta muy desbalanceado, se usa `class_weight="balanced"` para dar mas peso a la clase minoritaria, que en este caso son las maquinas con fallo.


In [19]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score


In [20]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(class_weight="balanced", random_state=26, max_iter=1000))
    ]
)


In [21]:
logistic_model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tra

In [22]:
y_pred_logistic = logistic_model.predict(X_test)
y_proba_logistic = logistic_model.predict_proba(X_test)[:, 1]


In [23]:
print(classification_report(y_test, y_pred_logistic))
 

              precision    recall  f1-score   support

           0       0.99      0.81      0.89      1932
           1       0.14      0.88      0.24        68

    accuracy                           0.81      2000
   macro avg       0.57      0.85      0.57      2000
weighted avg       0.97      0.81      0.87      2000



In [24]:
confusion_matrix(y_test, y_pred_logistic)


array([[1566,  366],
       [   8,   60]])

In [25]:
roc_auc_score(y_test, y_proba_logistic)


0.9248568992814518

**Interpretacion:** La Regresion Logistica consigue detectar la mayoria de maquinas con fallo, con un recall de 0.88 para la clase 1. Esto significa que identifica 60 de los 68 fallos reales del conjunto de prueba.

El principal problema es la baja precision en la clase fallo. Muchas maquinas sin fallo son marcadas como fallo, generando 366 falsas alarmas.

En un contexto de mantenimiento predictivo, este modelo podria ser util si se prioriza no dejar pasar fallos reales, pero tendria el inconveniente de provocar muchas revisiones innecesarias.

El ROC AUC es alto, lo que indica que el modelo separa bastante bien los casos de mayor y menor riesgo. Aun asi, sera necesario comparar con otros modelos para buscar un mejor equilibrio entre detectar fallos y reducir falsas alarmas.


## Modelo 2: Arbol de Decision

El segundo modelo probado es un Arbol de Decision.

Este modelo funciona creando reglas sobre las variables para separar los casos con fallo y sin fallo. Es facil de interpretar y puede detectar relaciones no lineales entre variables.

A diferencia de la Regresion Logistica, no depende tanto de relaciones lineales entre las variables. Sin embargo, puede sobreajustar si el arbol crece demasiado.

Como el dataset esta desbalanceado, tambien se usa `class_weight="balanced"` para dar mas importancia a la clase fallo.


In [26]:
from sklearn.tree import DecisionTreeClassifier


In [27]:
tree_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", DecisionTreeClassifier(
            class_weight="balanced",
            random_state=26,
            max_depth=5,
            min_samples_leaf=20
        ))
    ]
)


In [28]:
tree_model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tra

In [29]:
y_pred_tree = tree_model.predict(X_test)
y_proba_tree = tree_model.predict_proba(X_test)[:, 1]


In [30]:
print(classification_report(y_test, y_pred_tree))


              precision    recall  f1-score   support

           0       1.00      0.92      0.96      1932
           1       0.30      0.94      0.46        68

    accuracy                           0.92      2000
   macro avg       0.65      0.93      0.71      2000
weighted avg       0.97      0.92      0.94      2000



In [31]:
confusion_matrix(y_test, y_pred_tree)


array([[1785,  147],
       [   4,   64]])

In [32]:
roc_auc_score(y_test, y_proba_tree)


0.9301090001217879

**Interpretacion:** El Arbol de Decision mejora los resultados de la Regresion Logistica. Detecta 64 de los 68 fallos reales del conjunto de prueba, con un recall de 0.94 para la clase fallo.

Tambien reduce bastante las falsas alarmas: pasa de 366 falsos positivos con Regresion Logistica a 147 con el Arbol de Decision.

Aunque la precision de la clase fallo sigue siendo mejorable, el modelo consigue un equilibrio mas interesante entre detectar fallos y no marcar tantas maquinas sanas como problematicas.

**Decision:** El Arbol de Decision se convierte por ahora en el mejor modelo candidato, aunque todavia se comparara con modelos mas robustos como Random Forest y Gradient Boosting.


## Modelo 3: Random Forest

El tercer modelo probado es Random Forest.

Este modelo combina muchos arboles de decision para obtener predicciones mas estables. La idea es que un solo arbol puede ajustarse demasiado a los datos, mientras que varios arboles juntos suelen generalizar mejor.

Random Forest es un modelo muy usado en datos tabulares y puede detectar relaciones no lineales entre variables.

Como el dataset esta desbalanceado, se usa `class_weight="balanced"` para dar mas importancia a la clase fallo.


In [33]:
from sklearn.ensemble import RandomForestClassifier


In [34]:
forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=26,
            max_depth=6,
            min_samples_leaf=10,
            n_jobs=-1
        ))
    ]
)


In [35]:
forest_model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tra

In [36]:
y_pred_forest = forest_model.predict(X_test)
y_proba_forest = forest_model.predict_proba(X_test)[:, 1]


In [37]:
print(classification_report(y_test, y_pred_forest))


              precision    recall  f1-score   support

           0       1.00      0.94      0.97      1932
           1       0.34      0.93      0.49        68

    accuracy                           0.94      2000
   macro avg       0.67      0.93      0.73      2000
weighted avg       0.97      0.94      0.95      2000



In [38]:
confusion_matrix(y_test, y_pred_forest)


array([[1808,  124],
       [   5,   63]])

In [39]:
roc_auc_score(y_test, y_proba_forest)


0.9833759590792839

**Interpretacion:** Random Forest mejora el equilibrio general frente al Arbol de Decision. Detecta 63 de los 68 fallos reales y reduce las falsas alarmas a 124.

Aunque detecta un fallo menos que el Arbol de Decision, consigue una precision algo mejor para la clase fallo y un ROC AUC mucho mas alto.

Esto indica que el modelo es bueno diferenciando maquinas con mayor y menor riesgo de fallo.

**Decision:** Random Forest pasa a ser el mejor candidato por ahora, ya que mantiene un recall muy alto y reduce el numero de falsas alarmas respecto al Arbol de Decision.


## Modelo 4: Gradient Boosting

El cuarto modelo probado es Gradient Boosting.

Este modelo tambien se basa en arboles, pero los entrena de forma secuencial. Cada nuevo arbol intenta corregir errores cometidos por los anteriores.

Se prueba porque suele funcionar bien en datos tabulares y puede detectar relaciones complejas entre variables.

A diferencia de Random Forest, este modelo de scikit-learn no permite usar `class_weight` directamente, por lo que se evaluara si el desbalance de clases afecta a su capacidad para detectar fallos.


In [40]:
from sklearn.ensemble import GradientBoostingClassifier


In [41]:
boosting_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            random_state=26
        ))
    ]
)


In [42]:
boosting_model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tra

In [43]:
y_pred_boosting = boosting_model.predict(X_test)
y_proba_boosting = boosting_model.predict_proba(X_test)[:, 1]


In [44]:
print(classification_report(y_test, y_pred_boosting))


              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1932
           1       0.89      0.71      0.79        68

    accuracy                           0.99      2000
   macro avg       0.94      0.85      0.89      2000
weighted avg       0.99      0.99      0.99      2000



In [45]:
confusion_matrix(y_test, y_pred_boosting)


array([[1926,    6],
       [  20,   48]])

In [46]:
roc_auc_score(y_test, y_proba_boosting)


0.9908164352697599

**Interpretacion:** Gradient Boosting obtiene un comportamiento diferente a los modelos anteriores. Reduce mucho las falsas alarmas, con solo 6 maquinas sanas marcadas como fallo.

La precision de la clase fallo es alta, lo que significa que cuando el modelo predice fallo suele acertar. Sin embargo, el recall baja respecto a Random Forest, ya que deja sin detectar 20 de los 68 fallos reales.

Este modelo tiene el mejor ROC AUC hasta ahora, lo que indica que separa muy bien los casos de mayor y menor riesgo. Aun asi, con el umbral por defecto detecta menos fallos que Random Forest.

**Decision:** Gradient Boosting es un candidato fuerte si se quiere reducir falsas alarmas. Random Forest sigue siendo mejor si la prioridad es detectar la mayor cantidad posible de fallos.


## Modelo 5: KNN

El quinto modelo probado es KNN, que clasifica cada registro comparandolo con sus vecinos mas cercanos.

La idea es que una maquina se clasificara segun el comportamiento de otras maquinas parecidas.

Este modelo es sensible a la escala de las variables, por lo que el escalado numerico incluido en el preprocesamiento es especialmente importante.

Tambien puede verse afectado por el desbalance de clases, ya que hay muchas mas maquinas sin fallo que con fallo.


In [47]:
from sklearn.neighbors import KNeighborsClassifier


In [48]:
knn_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", KNeighborsClassifier(
            n_neighbors=5
        ))
    ]
)


In [50]:
knn_model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tra

In [51]:
y_pred_knn = knn_model.predict(X_test)
y_proba_knn = knn_model.predict_proba(X_test)[:, 1]


In [52]:
print(classification_report(y_test, y_pred_knn))


              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1932
           1       0.83      0.35      0.49        68

    accuracy                           0.98      2000
   macro avg       0.90      0.68      0.74      2000
weighted avg       0.97      0.98      0.97      2000



In [53]:
confusion_matrix(y_test, y_pred_knn)


array([[1927,    5],
       [  44,   24]])

In [54]:
roc_auc_score(y_test, y_proba_knn)


0.8685528559249787

**Interpretacion:** KNN genera muy pocas falsas alarmas, pero detecta pocos fallos reales. Solo identifica 24 de los 68 fallos del conjunto de prueba.

Esto puede deberse al fuerte desbalance de clases. Como hay muchas mas maquinas sin fallo, los vecinos mas cercanos tienden a pertenecer a la clase 0.

Aunque la precision de la clase fallo es alta, el recall es demasiado bajo para un problema de mantenimiento predictivo.

**Decision:** KNN no se considera un buen candidato final, ya que deja sin detectar demasiados fallos.


## Comparacion de modelos supervisados

Se comparan los modelos entrenados usando metricas centradas en la clase fallo.

En este problema no basta con mirar la accuracy, porque la clase fallo representa una parte pequena del dataset.

Por eso se revisan especialmente el recall, la precision, el F1-score, los falsos positivos y los falsos negativos.


In [55]:
from sklearn.metrics import precision_score, recall_score, f1_score


In [56]:
def evaluate_model(model_name, y_test, y_pred, y_proba):
    cm = confusion_matrix(y_test, y_pred)
    
    false_positives = cm[0, 1]
    false_negatives = cm[1, 0]
    
    return {
        "model": model_name,
        "precision_fallo": precision_score(y_test, y_pred, pos_label=1),
        "recall_fallo": recall_score(y_test, y_pred, pos_label=1),
        "f1_fallo": f1_score(y_test, y_pred, pos_label=1),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "falsos_positivos": false_positives,
        "falsos_negativos": false_negatives
    }


In [57]:
model_results = [
    evaluate_model("Logistic Regression", y_test, y_pred_logistic, y_proba_logistic),
    evaluate_model("Decision Tree", y_test, y_pred_tree, y_proba_tree),
    evaluate_model("Random Forest", y_test, y_pred_forest, y_proba_forest),
    evaluate_model("Gradient Boosting", y_test, y_pred_boosting, y_proba_boosting),
    evaluate_model("KNN", y_test, y_pred_knn, y_proba_knn)
]

results_df = pd.DataFrame(model_results)

results_df


,model,precision_fallo,recall_fallo,f1_fallo,roc_auc,falsos_positivos,falsos_negativos
0,Logistic Regression,0.140845,0.882353,0.242915,0.924857,366,8
1,Decision Tree,0.303318,0.941176,0.458781,0.930109,147,4
2,Random Forest,0.336898,0.926471,0.494118,0.983376,124,5
3,Gradient Boosting,0.888889,0.705882,0.786885,0.990816,6,20
4,KNN,0.827586,0.352941,0.494845,0.868553,5,44


In [58]:
results_df.sort_values(by="f1_fallo", ascending=False)


,model,precision_fallo,recall_fallo,f1_fallo,roc_auc,falsos_positivos,falsos_negativos
3,Gradient Boosting,0.888889,0.705882,0.786885,0.990816,6,20
4,KNN,0.827586,0.352941,0.494845,0.868553,5,44
2,Random Forest,0.336898,0.926471,0.494118,0.983376,124,5
1,Decision Tree,0.303318,0.941176,0.458781,0.930109,147,4
0,Logistic Regression,0.140845,0.882353,0.242915,0.924857,366,8


In [59]:
results_df.sort_values(by="recall_fallo", ascending=False)


,model,precision_fallo,recall_fallo,f1_fallo,roc_auc,falsos_positivos,falsos_negativos
1,Decision Tree,0.303318,0.941176,0.458781,0.930109,147,4
2,Random Forest,0.336898,0.926471,0.494118,0.983376,124,5
0,Logistic Regression,0.140845,0.882353,0.242915,0.924857,366,8
3,Gradient Boosting,0.888889,0.705882,0.786885,0.990816,6,20
4,KNN,0.827586,0.352941,0.494845,0.868553,5,44


**Interpretacion:** La comparacion muestra que no hay un unico modelo mejor en todo. Gradient Boosting consigue el mejor F1-score para la clase fallo y el ROC AUC mas alto, ademas de reducir mucho las falsas alarmas.

Decision Tree y Random Forest detectan mas fallos reales, pero generan muchas mas falsas alarmas. KNN tiene buena precision, pero deja escapar demasiados fallos.

**Decision:** Para este proyecto se selecciona Gradient Boosting como mejor modelo general, ya que ofrece el equilibrio mas solido entre detectar fallos y evitar falsas alarmas. Aun asi, se reconoce que Random Forest podria ser una alternativa si la prioridad principal fuera detectar el mayor numero posible de fallos.


## Ajuste del umbral de decision

Gradient Boosting obtiene buenos resultados, pero con el umbral por defecto de 0.5 deja sin detectar algunos fallos.

En esta seccion se prueban distintos umbrales de decision usando las probabilidades del modelo.

Bajar el umbral puede ayudar a detectar mas fallos, aunque normalmente aumenta el numero de falsas alarmas.


In [60]:
thresholds = [0.2, 0.3, 0.4, 0.5]

threshold_results = []

for threshold in thresholds:
    y_pred_threshold = (y_proba_boosting >= threshold).astype(int)
    
    cm = confusion_matrix(y_test, y_pred_threshold)
    
    threshold_results.append({
        "threshold": threshold,
        "precision_fallo": precision_score(y_test, y_pred_threshold, pos_label=1),
        "recall_fallo": recall_score(y_test, y_pred_threshold, pos_label=1),
        "f1_fallo": f1_score(y_test, y_pred_threshold, pos_label=1),
        "falsos_positivos": cm[0, 1],
        "falsos_negativos": cm[1, 0]
    })

threshold_results_df = pd.DataFrame(threshold_results)

threshold_results_df


,threshold,precision_fallo,recall_fallo,f1_fallo,falsos_positivos,falsos_negativos
0,0.2,0.700000,0.823529,0.756757,24,12
1,0.3,0.800000,0.823529,0.811594,14,12
2,0.4,0.836066,0.750000,0.790698,10,17
3,0.5,0.888889,0.705882,0.786885,6,20


**Interpretacion:** Al ajustar el umbral de decision de Gradient Boosting se observa que el umbral por defecto de 0.5 no es necesariamente el mejor para este problema.

Con un umbral de 0.3, el modelo detecta mas fallos reales que con 0.5, reduciendo los falsos negativos de 20 a 12. A cambio, las falsas alarmas aumentan de 6 a 14, pero siguen siendo un numero bajo.

Este ajuste mejora el F1-score de la clase fallo y ofrece un mejor equilibrio entre precision y recall.

**Decision:** Se selecciona Gradient Boosting con umbral 0.3 como modelo final, ya que permite detectar mas fallos sin generar demasiadas falsas alarmas.


In [61]:
final_threshold = 0.3

y_pred_final = (y_proba_boosting >= final_threshold).astype(int)


In [62]:
print(classification_report(y_test, y_pred_final))
confusion_matrix(y_test, y_pred_final)


              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1932
           1       0.80      0.82      0.81        68

    accuracy                           0.99      2000
   macro avg       0.90      0.91      0.90      2000
weighted avg       0.99      0.99      0.99      2000



array([[1918,   14],
       [  12,   56]])

**Interpretacion final del modelo:** Con el umbral ajustado a 0.3, Gradient Boosting consigue un resultado equilibrado para la clase fallo.

La precision de la clase fallo es 0.80, lo que significa que cuando el modelo predice fallo suele acertar en la mayoria de casos. El recall es 0.82, por lo que detecta la mayor parte de los fallos reales.

Esto supone una mejora respecto al umbral por defecto, ya que se reducen los fallos no detectados sin aumentar demasiado las falsas alarmas.

**Decision:** Se selecciona este modelo como resultado final del proyecto, ya que ofrece un buen equilibrio entre detectar fallos y evitar alertas innecesarias.


## Ajuste de hiperparametros con GridSearchCV

Para cumplir con el proceso de mejora del modelo, se aplica `GridSearchCV` sobre Gradient Boosting.

El objetivo es probar varias combinaciones de hiperparametros y seleccionar la que obtenga mejor rendimiento.

Como el dataset esta desbalanceado y nos interesa equilibrar precision y recall en la clase fallo, se usa `f1` como metrica principal de busqueda.


In [69]:
from sklearn.model_selection import GridSearchCV


In [70]:
grid_boosting_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", GradientBoostingClassifier(random_state=26))
    ]
)


In [71]:
param_grid = {
    "model__n_estimators": [100, 200],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [2, 3]
}


In [72]:
grid_search = GridSearchCV(
    estimator=grid_boosting_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1
)


In [73]:
grid_search.fit(X_train, y_train)


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=26))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__learning_rate': [0.05, 0.1], 'model__max_depth': [2, 3], 'model__n_estimators': [100, 200]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed

In [74]:
grid_search.best_params_


{'model__learning_rate': 0.1,
 'model__max_depth': 3,
 'model__n_estimators': 200}

In [75]:
grid_search.best_score_


np.float64(0.6908579225652396)

In [76]:
best_boosting_model = grid_search.best_estimator_

y_proba_best_boosting = best_boosting_model.predict_proba(X_test)[:, 1]


In [77]:
y_pred_best_boosting = (y_proba_best_boosting >= 0.3).astype(int)

print(classification_report(y_test, y_pred_best_boosting))
confusion_matrix(y_test, y_pred_best_boosting)


              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1932
           1       0.75      0.82      0.78        68

    accuracy                           0.98      2000
   macro avg       0.87      0.91      0.89      2000
weighted avg       0.99      0.98      0.98      2000



array([[1913,   19],
       [  12,   56]])

In [78]:
roc_auc_score(y_test, y_proba_best_boosting)


0.990839270490805

**Interpretacion:** GridSearchCV prueba varias combinaciones de hiperparametros para Gradient Boosting. La mejor combinacion encontrada fue `learning_rate=0.1`, `max_depth=3` y `n_estimators=200`.

Al evaluar este modelo en test con umbral 0.3, mantiene un recall de 0.82 para la clase fallo, pero obtiene una precision algo menor que el modelo Gradient Boosting anterior.

El modelo ajustado con GridSearch genera 19 falsas alarmas, frente a las 14 del modelo anterior, manteniendo los mismos 12 fallos no detectados.

**Decision:** Aunque GridSearchCV cumple el proceso de ajuste de hiperparametros, no mejora el resultado final. Por este motivo, se mantiene como modelo final el Gradient Boosting inicial con umbral 0.3.


## Modelo no supervisado: Isolation Forest

Ademas de los modelos supervisados, se prueba un enfoque no supervisado con Isolation Forest.

El objetivo no es predecir directamente `Machine failure`, sino detectar registros con comportamiento anomalo a partir de las variables operativas.

Este enfoque tiene sentido en mantenimiento predictivo porque, en una empresa real, no siempre se dispone de etiquetas claras de fallo. Un sistema de deteccion de anomalias podria servir como apoyo para identificar maquinas que se comportan de forma diferente al patron normal.


In [63]:
from sklearn.ensemble import IsolationForest


In [64]:
isolation_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", IsolationForest(
            contamination=0.034,
            random_state=26
        ))
    ]
)


In [65]:
isolation_model.fit(X_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tra

In [66]:
isolation_pred = isolation_model.predict(X_test)


In [67]:
isolation_anomaly = (isolation_pred == -1).astype(int)


In [68]:
print(classification_report(y_test, isolation_anomaly))
confusion_matrix(y_test, isolation_anomaly)


              precision    recall  f1-score   support

           0       0.97      0.97      0.97      1932
           1       0.12      0.13      0.13        68

    accuracy                           0.94      2000
   macro avg       0.55      0.55      0.55      2000
weighted avg       0.94      0.94      0.94      2000



array([[1868,   64],
       [  59,    9]])

**Interpretacion:** Isolation Forest se prueba como modelo no supervisado para detectar comportamientos anomalos sin usar la variable `Machine failure` durante el entrenamiento.

Los resultados muestran que solo detecta 9 de los 68 fallos reales del conjunto de prueba. Esto indica que, en este dataset, muchos fallos no aparecen como anomalias claras respecto al resto de registros.

**Decision:** El modelo no supervisado se mantiene como analisis complementario, pero no se considera adecuado como modelo principal. Para este proyecto, los modelos supervisados son mas utiles porque existe una variable objetivo clara y permiten aprender directamente de los fallos registrados.


## Guardado del modelo final

Se guarda el modelo final junto con el umbral seleccionado para poder reutilizarlo mas adelante sin volver a entrenarlo.


In [79]:
import joblib


In [80]:
final_model_artifact = {
    "model": boosting_model,
    "threshold": 0.3,
    "features": features
}


In [81]:
joblib.dump(final_model_artifact, "../models/final_model.pkl")


['../models/final_model.pkl']

In [82]:
loaded_model_artifact = joblib.load("../models/final_model.pkl")

loaded_model_artifact.keys()


dict_keys(['model', 'threshold', 'features'])